In [12]:
import sys
import logging
import datetime
import math
from typing import Dict, List, Tuple, Any
import requests
from requests.adapters import HTTPAdapter
from urllib3.util import Retry
import pandas as pd
import numpy as np

# Resolve problemas de codificação no console
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
if hasattr(sys.stderr, 'reconfigure'):
    sys.stderr.reconfigure(encoding='utf-8')

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def obter_client_http_robusto() -> requests.Session:
    session = requests.Session()
    retry_strategy = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504], allowed_methods=["GET"])
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'application/json'
    })
    return session

def buscar_dados_copa(session: requests.Session) -> List[Dict[str, Any]]:
    url = "https://site.api.espn.com/apis/site/v2/sports/soccer/fifa.world/scoreboard?dates=20260611-20260719&limit=100"
    logger.info("Realizando Web Scraping dos dados da ESPN...")
    try:
        response = session.get(url, timeout=15)
        response.raise_for_status()
        data = response.json()
        fixtures = []
        for event in data.get('events', []):
            try:
                competitors = event['competitions'][0]['competitors']
                home_team = next(c for c in competitors if c['homeAway'] == 'home')
                away_team = next(c for c in competitors if c['homeAway'] == 'away')
                status_espn = event['status']['type']['state']

                if status_espn == 'post': status_short = 'FT'
                elif status_espn == 'in': status_short = '2H'
                else: status_short = 'NS'

                fixture = {
                    "fixture": {"date": event['date'], "status": {"short": status_short, "long": event['status']['type']['description']}},
                    "teams": {"home": {"name": home_team['team']['name']}, "away": {"name": away_team['team']['name']}},
                    "goals": {"home": int(home_team.get('score', 0)) if status_short != 'NS' else None, "away": int(away_team.get('score', 0)) if status_short != 'NS' else None}
                }
                fixtures.append(fixture)
            except Exception: continue
        return fixtures
    except requests.exceptions.RequestException as e:
        logger.error(f"Falha de rede: {e}")
        raise

RANKING_FIFA_OFICIAL = {
    'Argentina': 1, 'Spain': 2, 'France': 3, 'England': 4, 'Portugal': 5,
    'Brazil': 6, 'Morocco': 7, 'Netherlands': 8, 'Belgium': 9, 'Germany': 10,
    'Croatia': 11, 'Italy': 12, 'Colombia': 13, 'Mexico': 14, 'Senegal': 15,
    'Uruguay': 16, 'United States': 17, 'Japan': 18, 'Switzerland': 19, 'Iran': 20,
    'Denmark': 21, 'Türkiye': 22, 'Ecuador': 23, 'Austria': 24, 'South Korea': 25,
    'Nigeria': 26, 'Australia': 27, 'Algeria': 28, 'Wales': 29, 'Egypt': 30,
    'Peru': 31, 'Serbia': 32, 'Russia': 33, 'Czechia': 34, 'Qatar': 35,
    'Ivory Coast': 38, 'Scotland': 39, 'Tunisia': 41, 'Panama': 45, 'Canada': 49,
    'Saudi Arabia': 53, 'Paraguay': 56, 'Iraq': 58, 'South Africa': 59,
    'Congo DR': 63, 'Uzbekistan': 64, 'Cape Verde': 65, 'Ghana': 68,
    'Jordan': 71, 'Bosnia-Herzegovina': 74, 'Haiti': 90, 'Curaçao': 91,
    'New Zealand': 104
}

MAPEAMENTO_NOMES_ESPN = {
    'USA': 'United States', 'Korea Republic': 'South Korea', 'Democratic Republic of the Congo': 'Congo DR',
    'Bosnia and Herzegovina': 'Bosnia-Herzegovina', 'Turkey': 'Türkiye', 'Czech Republic': 'Czechia'
}

def normalizar_nome(team: str) -> str:
    return MAPEAMENTO_NOMES_ESPN.get(team, team)

def obter_elo_inicial_por_retrospecto(team: str) -> float:
    team_norm = normalizar_nome(team)
    rank = RANKING_FIFA_OFICIAL.get(team_norm, 80)
    return 2100.0 - (150.0 * math.log(rank))

def calcular_elo_rating(fixtures: List[Dict[str, Any]]) -> Tuple[Dict[str, float], float]:
    K_FACTOR = 40
    elos = {team: obter_elo_inicial_por_retrospecto(team) for team in RANKING_FIFA_OFICIAL.keys()}
    status_encerrados = ['FT', 'AET', 'PEN']
    jogos_encerrados = [f for f in fixtures if f.get('fixture', {}).get('status', {}).get('short') in status_encerrados]
    jogos_encerrados.sort(key=lambda x: x.get('fixture', {}).get('date', ''))
    total_gols, total_partidas = 0, max(len(jogos_encerrados), 1)

    for jogo in jogos_encerrados:
        try:
            t_casa, t_fora = normalizar_nome(jogo['teams']['home']['name']), normalizar_nome(jogo['teams']['away']['name'])
            g_casa, g_fora = jogo['goals']['home'], jogo['goals']['away']
            total_gols += (g_casa + g_fora)

            if t_casa not in elos: elos[t_casa] = obter_elo_inicial_por_retrospecto(t_casa)
            if t_fora not in elos: elos[t_fora] = obter_elo_inicial_por_retrospecto(t_fora)

            exp_casa = 1 / (1 + 10 ** ((elos[t_fora] - elos[t_casa]) / 400))
            exp_fora = 1 / (1 + 10 ** ((elos[t_casa] - elos[t_fora]) / 400))

            s_casa, s_fora = (1.0, 0.0) if g_casa > g_fora else ((0.0, 1.0) if g_casa < g_fora else (0.5, 0.5))
            multiplicador_margem = math.log(abs(g_casa - g_fora) + 1) if abs(g_casa - g_fora) > 0 else 1.0

            elos[t_casa] += (K_FACTOR * multiplicador_margem * (s_casa - exp_casa))
            elos[t_fora] += (K_FACTOR * multiplicador_margem * (s_fora - exp_fora))
        except Exception: continue

    # Ancoragem Histórica Bayesiana para o formato de 48 equipas (Média Moderna: ~2.7 golos/jogo)
    media_real_torneio = total_gols / (total_partidas * 2) if total_partidas > 0 else 1.35
    media_global_gols = (media_real_torneio * 0.4) + (1.35 * 0.6) # Peso na média de 1.35 por equipa

    return elos, media_global_gols

def ajuste_dixon_coles(x: int, y: int, lambda_c: float, lambda_f: float, rho: float = 0.18) -> float:
    if x == 0 and y == 0: return max(0.0, 1 - lambda_c * lambda_f * rho)
    elif x == 0 and y == 1: return 1 + lambda_c * rho
    elif x == 1 and y == 0: return 1 + lambda_f * rho
    elif x == 1 and y == 1: return max(0.0, 1 - rho)
    return 1.0

def prever_partida_quant(time_casa: str, time_fora: str, elos: Dict[str, float], media_global: float) -> Tuple[float, float, float, float, float, List[Tuple[str, float]], float, float]:
    t_casa_norm, t_fora_norm = normalizar_nome(time_casa), normalizar_nome(time_fora)
    elo_c = elos.get(t_casa_norm, obter_elo_inicial_por_retrospecto(t_casa_norm))
    elo_f = elos.get(t_fora_norm, obter_elo_inicial_por_retrospecto(t_fora_norm))

    # Fator calibrado: permite que os favoritos disparem em jogos fáceis (xG 2.5 a 3.5), gerando 2x0 e 3x0
    FATOR_COMPRESSAO = 0.00165
    lambda_casa = media_global * math.exp((elo_c - elo_f) * FATOR_COMPRESSAO)
    lambda_fora = media_global * math.exp((elo_f - elo_c) * FATOR_COMPRESSAO)

    # Limites mais amplos para permitir goleadas raras, mas possíveis
    lambda_casa, lambda_fora = np.clip(lambda_casa, 0.1, 4.5), np.clip(lambda_fora, 0.1, 4.5)

    dc_prob_casa = dc_prob_empate = dc_prob_fora = 0.0
    prob_over_25 = prob_btts = 0.0
    placares_probs = []

    def poisson_pmf(k, lam): return (lam**k * math.exp(-lam)) / math.factorial(k)

    for x in range(8):
        for y in range(8):
            prob_pura = poisson_pmf(x, lambda_casa) * poisson_pmf(y, lambda_fora)
            # Retranca ajustada para 0.18: mantém empates reais (1-1), mas não estrangula o 2-1
            prob_ajustada = prob_pura * ajuste_dixon_coles(x, y, lambda_casa, lambda_fora, rho=0.18)

            placares_probs.append((f"{x} x {y}", prob_ajustada))

            if x > y: dc_prob_casa += prob_ajustada
            elif x == y: dc_prob_empate += prob_ajustada
            else: dc_prob_fora += prob_ajustada

            if (x + y) > 2.5: prob_over_25 += prob_ajustada
            if x > 0 and y > 0: prob_btts += prob_ajustada

    # Ordena e extrai o Top 3 placares exatos
    placares_probs.sort(key=lambda item: item[1], reverse=True)
    top_3_placares = placares_probs[:3]

    total_adj = dc_prob_casa + dc_prob_empate + dc_prob_fora
    return (dc_prob_casa/total_adj, dc_prob_empate/total_adj, dc_prob_fora/total_adj, lambda_casa, lambda_fora, top_3_placares, prob_over_25/total_adj, prob_btts/total_adj)

def calcular_fair_odds(prob: float) -> str:
    return "+99.00" if prob <= 0.01 else f"{1 / prob:.2f}"

def formatar_data_local(data_iso: str) -> str:
    dt_local = datetime.datetime.fromisoformat(data_iso.replace('Z', '+00:00')).astimezone()
    return dt_local.strftime('%d/%m %H:%M')

def main():
    logger.info("Iniciando Motor Quantitativo de Previsão...")
    session = obter_client_http_robusto()
    try:
        fixtures = buscar_dados_copa(session)
        elos_times, media_global = calcular_elo_rating(fixtures)

        jogos_futuros = [f for f in fixtures if f.get('fixture', {}).get('status', {}).get('short') == 'NS']
        jogos_futuros.sort(key=lambda x: x.get('fixture', {}).get('date', ''))

        print(f"\n🏟️ --- PREVISÕES QUANTITATIVAS DEFINITIVAS ---\n")

        for jogo in jogos_futuros:
            try:
                time_casa = jogo['teams']['home']['name']
                time_fora = jogo['teams']['away']['name']

                # Ignora jogos de mata-mata genéricos enviados pela API antecipadamente
                palavras_ignorar = ['Winner', 'Place', 'Round', 'Group']
                if any(p in time_casa or p in time_fora for p in palavras_ignorar): continue

                data_jogo_str = formatar_data_local(jogo['fixture']['date'])
                p_c, p_e, p_f, exp_c, exp_f, top_placares, p_over, p_btts = prever_partida_quant(time_casa, time_fora, elos_times, media_global)

                # LÓGICA DE ZONA DE EMPATE (THRESHOLD DE 12%)
                margem_vitoria = abs(p_c - p_f)
                if margem_vitoria < 0.12: palpite = "Empate Técnico"
                elif p_c > p_f: palpite = f"Vitória de {time_casa}"
                else: palpite = f"Vitória de {time_fora}"

                str_top_placares = " | ".join([f"{p[0]} ({p[1]*100:.1f}%)" for p in top_placares])
                t_casa_n, t_fora_n = normalizar_nome(time_casa), normalizar_nome(time_fora)

                print(f"🗓️ {data_jogo_str} | {time_casa} vs {time_fora}")
                print(f"   📈 Elo Rating: {time_casa} ({elos_times.get(t_casa_n, 1500):.0f}) vs {time_fora} ({elos_times.get(t_fora_n, 1500):.0f})")
                print(f"   📊 Probs: {time_casa} ({p_c*100:.1f}%) | Empate ({p_e*100:.1f}%) | {time_fora} ({p_f*100:.1f}%)")
                print(f"   🎯 Top 3 Placares: {str_top_placares} (xG: {exp_c:.2f} x {exp_f:.2f})")
                print(f"   🎲 Mercados Base: Over 2.5 ({p_over*100:.1f}%) | Ambas Marcam ({p_btts*100:.1f}%)")
                print(f"   🔥 Palpite de Valor: {palpite}")
                print("-" * 65)
            except Exception as e: logger.error(f"Erro no jogo {time_casa}: {e}")
    except Exception as e: logger.critical(f"Falha no pipeline: {e}")

if __name__ == "__main__":
    main()


🏟️ --- PREVISÕES QUANTITATIVAS DEFINITIVAS ---

🗓️ 23/06 03:00 | Jordan vs Algeria
   📈 Elo Rating: Jordan (1448) vs Algeria (1597)
   📊 Probs: Jordan (25.1%) | Empate (19.1%) | Algeria (55.8%)
   🎯 Top 3 Placares: 0 x 1 (11.7%) | 1 x 2 (9.8%) | 1 x 1 (8.9%) (xG: 1.11 x 1.81)
   🎲 Mercados Base: Over 2.5 (55.9%) | Ambas Marcam (54.2%)
   🔥 Palpite de Valor: Vitória de Algeria
-----------------------------------------------------------------
🗓️ 23/06 17:00 | Portugal vs Uzbekistan
   📈 Elo Rating: Portugal (1843) vs Uzbekistan (1467)
   📊 Probs: Portugal (78.3%) | Empate (11.9%) | Uzbekistan (9.7%)
   🎯 Top 3 Placares: 2 x 0 (11.6%) | 3 x 0 (10.2%) | 1 x 0 (10.0%) (xG: 2.64 x 0.76)
   🎲 Mercados Base: Over 2.5 (65.8%) | Ambas Marcam (48.4%)
   🔥 Palpite de Valor: Vitória de Portugal
-----------------------------------------------------------------
🗓️ 23/06 20:00 | England vs Ghana
   📈 Elo Rating: England (1905) vs Ghana (1483)
   📊 Probs: England (82.1%) | Empate (10.3%) | Ghana (7.6%